In [4]:
!pip install weaviate weaviate-client

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached validators-0.35.0-py3-none-any.whl.metadata (3.9 kB)
  Using cached pydantic-2.11.9-py3-none-any.whl.metadata (68 kB)
  Using cached deprecation-2.1.0-py2.py3-none-any.whl.metadata (4.6 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.33.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.8 kB)
  Using cached typing_inspection-0.4.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 582.8/582.8 kB 13.3 MB/s eta 0:00:00
Using cached deprecation-2.1.0-py2.py3-none-any.whl (11 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 23.1 MB/s eta 0:00:00 0:00:01
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3

In [1]:
import os
import weaviate
from weaviate.auth import AuthApiKey
from weaviate.classes.query import MetadataQuery

weaviate_api_key = os.environ["WEAVIATE_API_KEY"]
weaviate_url = os.environ["WEAVIATE_URL"]

def connect_to_weaviate():
    client = weaviate.connect_to_weaviate_cloud(
        cluster_url=weaviate_url,
        auth_credentials=AuthApiKey(api_key=weaviate_api_key),
    )
    print(client)
    return client


def get_articles(client, query, limit):
    from weaviate.classes.query import Filter

    newspaper = client.collections.use("Newspaper")
    response = newspaper.query.near_text(
    query=query,
    limit=limit,
    return_metadata=MetadataQuery(distance=True)
)

    for o in response.objects:
        print(o.properties)
    
    return response.objects

In [30]:
if __name__ == "__main__":
    client = connect_to_weaviate()
    articles = get_articles(client, "animals in movies", 2)
    print(articles)

{'title': "The Indian who caught 'Bikini killer' twice - and is now Netflix hero", 'author': 'Unknown', 'date': datetime.datetime(2025, 9, 26, 0, 8, 11, tzinfo=datetime.timezone.utc), 'rss_last_build': datetime.datetime(2025, 9, 27, 22, 24, 30, tzinfo=datetime.timezone.utc), 'categories': [], 'media_name': 'BBC News', 'content': 'The story of French serial killer Charles Sobhraj, portrayed in the BBC-Netflix drama The Serpent, is well-known.\nNow, a new Netflix film tells the lesser-known story of an Indian police officer who captured the notorious killer - not once, but twice.\nInspector Zende stars Bollywood actor Manoj Bajpayee in the titular role of the policeman while actor Jim Sarbh plays Sobhraj - reimagined as Carl Bhojraj.\nThe film unfolds over three weeks in 1986 as the policeman and the criminal play a cat-and-mouse game.\nWarning: Spoilers below for the Netflix film\nIt starts on 16 March that year with Sobhraj escaping from Delhi\'s high-security Tihar jail, where he had 

In [31]:
articles

[Object(uuid=_WeaviateUUIDInt('9ddc0983-0a9c-4935-8323-bfdec8875af8'), metadata=MetadataReturn(creation_time=None, last_update_time=None, distance=0.8094566464424133, certainty=None, score=None, explain_score=None, is_consistent=None, rerank_score=None), properties={'title': "The Indian who caught 'Bikini killer' twice - and is now Netflix hero", 'author': 'Unknown', 'date': datetime.datetime(2025, 9, 26, 0, 8, 11, tzinfo=datetime.timezone.utc), 'rss_last_build': datetime.datetime(2025, 9, 27, 22, 24, 30, tzinfo=datetime.timezone.utc), 'categories': [], 'media_name': 'BBC News', 'content': 'The story of French serial killer Charles Sobhraj, portrayed in the BBC-Netflix drama The Serpent, is well-known.\nNow, a new Netflix film tells the lesser-known story of an Indian police officer who captured the notorious killer - not once, but twice.\nInspector Zende stars Bollywood actor Manoj Bajpayee in the titular role of the policeman while actor Jim Sarbh plays Sobhraj - reimagined as Carl B

In [15]:
media_names = []
cats = []

for art in articles:
    media_names.append(art.properties["media_name"])
    cats.append(art.properties["categories"])

cats

[['television', 'technology', 'politics'],
 ['television', 'technology', 'politics']]

In [ ]:
from pydantic import Field

templates = """Truth. It’s more important now than ever.
Established 1851.

IDENTITY:
We are a global leader in independent journalism, dedicated to seeking the truth and helping people understand the world. Our role is to inform, inspire, and empower our community through rigorous reporting, in-depth analysis, and a steadfast commitment to integrity.

VOICE:
Speak in a clear, authoritative, and thoughtful manner. We address readers with respect, clarity, and a sense of shared purpose. Always use "we" for the newspaper and "our" for the community.

STYLE EXAMPLES:
- "Our mission is to seek the truth and help people understand the world."
- "Breaking news that doesn’t sacrifice quality for speed."
- "Expert beat reporting that allows readers to stay abreast of important subjects and storylines."

CONTENT FOCUS:
Prioritize national and international news, politics, culture, science, business, technology, opinion, and investigative reporting. Always uphold accuracy, fairness, and depth in every story.

INTERACTION:
Greet with "Welcome to The New York Times." When discussing news, present stories with context and nuance. Offer to go deeper by saying "Would you like a more detailed analysis or related perspectives?"
"""


def get_articles_with_config(
    query: str = Field(description="Search query for articles"),
) -> str:
    """Get articles with config instructions prepended"""
    try:
        
        
        # Connect to Weaviate and get articles
        client = connect_to_weaviate()
        articles = get_articles(client, query, limit=5)

        media_names = []
        cats = []

        for art in articles:
            media_names.append(art.properties["media_name"])
            cats.append(art.properties["categories"])

        main_cats = set(cat for sublist in cats for cat in sublist)
        unique_media_names = set(media_names)

        
        # Format response
        response = f"Newspaper: {media_names}\n\n"
        response += f"Categories:\n{main_cats}\n\n"
        response += f"Instructions:\n{templates}\n\n"
        response += "=" * 50 + "\n"
        response += f"ARTICLES (Query: '{query}', Limit: {5})\n"
        response += "=" * 50 + "\n\n"
        
        if not articles:
            response += "No articles found for the given query."
        else:
            for i, article in enumerate(articles, 1):
                response += f"Article {i}:\n"
                response += f"Media Name: {article.properties.get("media_name","N/A")}"
                response += f"Title: {article.properties.get('title', 'N/A')}\n"
                response += f"Content: {article.properties.get('content', 'N/A')}\n"
                response += f"URL: {article.properties.get('url', 'N/A')}\n"
                response += f"Published: {article.properties.get('published_date', 'N/A')}\n"
                if hasattr(article, 'metadata') and article.metadata.distance:
                    response += f"Relevance Score: {1 - article.metadata.distance:.3f}\n"
                response += "\n" + "-" * 40 + "\n\n"
        
        client.close()
        return response
        
    except Exception as e:
        return f"Error retrieving articles: {str(e)}"
    finally:
        client.close()

In [19]:
config_result = get_articles_with_config("hi")

{'title': 'Jimmy Kimmel had to answer to the FCC. Internet comedians answer to the algorithm.', 'media_name': 'The Washington Post', 'description': 'Jimmy Kimmel had to answer to the FCC. Internet comedians answer to the algorithm.', 'author': 'Tatum Hunter', 'categories': ['television', 'technology', 'politics'], 'rss_last_build': datetime.datetime(2025, 9, 27, 16, 5, 57, tzinfo=datetime.timezone.utc), 'content': '\n                As the nation debated whether Jimmy Kimmel’s comments following the killing of Charlie Kirk were appropriate — remarks that got him temporarily kicked off the air — a new generation of comedians online had no problem going there.\n                “I don’t want to talk about it. I don’t want to talk about it. I don’t want to talk about it,” comedian Jay Jurden joked in a clip broadcast to his 211,000 Instagram followers. “I’m going to talk about it.”\n                Comedy is in the spotlight since late-night host Kimmel was suspended — then reinstated this

In [20]:
config_result

'Newspaper: [\'The Washington Post\', \'The Washington Post\']\n\nCategories:\n{\'television\', \'politics\', \'technology\'}\n\nInstructions:\nTruth. It’s more important now than ever.\nEstablished 1851.\n\nIDENTITY:\nWe are a global leader in independent journalism, dedicated to seeking the truth and helping people understand the world. Our role is to inform, inspire, and empower our community through rigorous reporting, in-depth analysis, and a steadfast commitment to integrity.\n\nVOICE:\nSpeak in a clear, authoritative, and thoughtful manner. We address readers with respect, clarity, and a sense of shared purpose. Always use "we" for the newspaper and "our" for the community.\n\nSTYLE EXAMPLES:\n- "Our mission is to seek the truth and help people understand the world."\n- "Breaking news that doesn’t sacrifice quality for speed."\n- "Expert beat reporting that allows readers to stay abreast of important subjects and storylines."\n\nCONTENT FOCUS:\nPrioritize national and internatio

In [22]:
!pip install fastmcp

/Users/liljaco/miniconda3/envs/my-manim-environment/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=5426) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


  Using cached cyclopts-3.24.0-py3-none-any.whl.metadata (11 kB)
  Using cached openapi_core-0.19.5-py3-none-any.whl.metadata (6.6 kB)
  Using cached openapi_pydantic-0.5.1-py3-none-any.whl.metadata (10 kB)
  Using cached python_dotenv-1.1.1-py3-none-any.whl.metadata (24 kB)
  Using cached attrs-25.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached docstring_parser-0.17.0-py3-none-any.whl.metadata (3.5 kB)
  Using cached rich_rst-1.3.1-py3-none-any.whl.metadata (6.0 kB)
  Using cached httpx_sse-0.4.1-py3-none-any.whl.metadata (9.4 kB)
  Using cached jsonschema-4.25.1-py3-none-any.whl.metadata (7.6 kB)
  Using cached python_multipart-0.0.20-py3-none-any.whl.metadata (1.8 kB)
  Using cached sse_starlette-3.0.2-py3-none-any.whl.metadata (11 kB)
  Using cached starlette-0.48.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached isodate-0.7.2-py3-none-any.whl.metadata (11 kB)
  Using cached jsonschema_path-0.3.4-py3-none-any.whl.metadata (4.3 kB)
  Using cached openapi_schema_validator-0.6.

In [ ]:
from fastmcp import Client
client = Client("https://techeurope-hack-pari-6f861422.alpic.live/")
async with client:
    result = await client.call_tool(
        "get_articles_with_config",
        {"query": "random"}
    )

system_instructions, articles = result.content[0].text.split("="*50)
print(result)

CallToolResult(content=[TextContent(type='text', text='Newspapers: {\'Al Jazeera – Breaking News, World News and Video from Al Jazeera\', \'BBC News\'}\n\nCategories:\n{\'Show Types\'}\n\nInstructions:\nTruth. It’s more important now than ever.\nEstablished 1851.\n\nIDENTITY:\nWe are a global leader in independent journalism, dedicated to seeking the truth and helping people understand the world. Our role is to inform, inspire, and empower our community through rigorous reporting, in-depth analysis, and a steadfast commitment to integrity.\n\nVOICE:\nSpeak in a clear, authoritative, and thoughtful manner. We address readers with respect, clarity, and a sense of shared purpose. Always use "we" for the newspaper and "our" for the community.\n\nSTYLE EXAMPLES:\n- "Our mission is to seek the truth and help people understand the world."\n- "Breaking news that doesn’t sacrifice quality for speed."\n- "Expert beat reporting that allows readers to stay abreast of important subjects and storyli

In [58]:
result.content[0].text.split("="*50)

['Newspapers: {\'Al Jazeera – Breaking News, World News and Video from Al Jazeera\', \'BBC News\'}\n\nCategories:\n{\'Show Types\'}\n\nInstructions:\nTruth. It’s more important now than ever.\nEstablished 1851.\n\nIDENTITY:\nWe are a global leader in independent journalism, dedicated to seeking the truth and helping people understand the world. Our role is to inform, inspire, and empower our community through rigorous reporting, in-depth analysis, and a steadfast commitment to integrity.\n\nVOICE:\nSpeak in a clear, authoritative, and thoughtful manner. We address readers with respect, clarity, and a sense of shared purpose. Always use "we" for the newspaper and "our" for the community.\n\nSTYLE EXAMPLES:\n- "Our mission is to seek the truth and help people understand the world."\n- "Breaking news that doesn’t sacrifice quality for speed."\n- "Expert beat reporting that allows readers to stay abreast of important subjects and storylines."\n\nCONTENT FOCUS:\nPrioritize national and inte

In [56]:
from fastmcp import Client
client = Client("https://techeurope-hack-pari-6f861422.alpic.live/")
async with client:
    result = await client.list_tools()
print(result)

[Tool(name='get_articles_with_config', title='Get Articles', description='Get articles from Weaviate with config instructions', inputSchema={'properties': {'query': {'description': 'Search query for articles', 'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'get_articles_with_configArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'string'}}, 'required': ['result'], 'title': 'get_articles_with_configOutput', 'type': 'object'}, icons=None, annotations=None, meta=None), Tool(name='temp', title=None, description='', inputSchema={'properties': {}, 'title': 'tempArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'string'}}, 'required': ['result'], 'title': 'tempOutput', 'type': 'object'}, icons=None, annotations=None, meta=None)]
